# Thesis Master — Phase 2 Training (Google Colab)

**Zero-upload approach:** code comes from GitHub, DSA data is downloaded directly from UCI (~3 MB zip).

## Before you start
1. Make your GitHub repo **public** (or create a Personal Access Token for private repos).
2. Set your repo URL in the **Configuration** cell below.
3. In Colab → **Runtime → Change runtime type → T4 GPU (free)**.
4. Run cells **top to bottom**.

## Resuming after a disconnection
Results are saved to **Google Drive** after every experiment.
When you reconnect, re-run all cells from the top — completed jobs are skipped automatically.
The processed DSA data is also cached to Drive so preprocessing only runs once.

## Step 0 — Configuration (edit this cell)

In [ ]:
# ── EDIT THESE ───────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/TimosEle23/Thesis_Master.git"  # ← your repo
GIT_BRANCH    = "phase2-colab"   # branch with the 170-job Phase 2 code
GDRIVE_DIR    = "/content/drive/MyDrive/ThesisMaster"                  # Drive folder
# For a private repo, use a token:
# GITHUB_REPO = "https://YOUR_TOKEN@github.com/YOUR_USERNAME/Thesis_Master.git"
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR    = "/content/thesis"
RESULTS_DIR    = f"{GDRIVE_DIR}/results"
DSA_CACHE_DIR  = f"{GDRIVE_DIR}/dsa_processed"   # processed DSA cached here
LOG_PATH       = f"{GDRIVE_DIR}/phase2_run.log"

import os
print("Config OK")
print(f"  Repo         : {GITHUB_REPO}")
print(f"  Project dir  : {PROJECT_DIR}")
print(f"  Results dir  : {RESULTS_DIR}")

## Step 1 — Check GPU

In [ ]:
import subprocess, torch

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True
)
print("GPU:", result.stdout.strip() if result.returncode == 0 else "❌ No GPU — switch to GPU runtime!")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} ({torch.version.cuda})")
if torch.cuda.is_available():
    print(f"Device  : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  Go to Runtime → Change runtime type → GPU (T4) and restart!")

## Step 2 — Mount Drive + clone repo

In [ ]:
import os, subprocess, sys

# Mount Drive (for saving results + cached data)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RESULTS_DIR + "/phase2", exist_ok=True)
print("Drive mounted ✅")

# Clone (or update) the repo
if os.path.isdir(f"{PROJECT_DIR}/.git"):
    print("Repo already cloned — pulling latest changes...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", GIT_BRANCH], check=True)
else:
    print(f"Cloning {GITHUB_REPO} ...")
    subprocess.run(["git", "clone", "--depth=1", "--branch", GIT_BRANCH, GITHUB_REPO, PROJECT_DIR], check=True)

# Confirm key directories
for d in ["src", "scripts", "configs"]:
    status = "✅" if os.path.isdir(f"{PROJECT_DIR}/{d}") else "❌ MISSING"
    print(f"  {status}  {d}/")

# Add project to path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print("\nRepo ready ✅")

## Step 3 — Install dependencies

In [ ]:
import subprocess, sys, torch

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# PyTorch Geometric
print("Installing torch_geometric...")
pip("torch_geometric")

# Optional C++ extensions (speed up graph ops — skip if wheels missing)
torch_ver  = torch.__version__.split("+")[0]
cuda_short = torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
pyg_url    = f"https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_short}.html"
print(f"Installing torch_scatter / torch_sparse from {pyg_url} ...")
try:
    pip("torch_scatter", "torch_sparse", "-f", pyg_url)
    print("  torch_scatter / torch_sparse ✅")
except subprocess.CalledProcessError:
    print("  Not found for this combo — skipping (not strictly required)")

# Other packages
print("Installing omegaconf, aeon, jsonargparse, tensorboard...")
pip("omegaconf", "aeon", "jsonargparse", "tensorboard")

print("\nAll dependencies installed ✅")

## Step 4 — Get DSA data

Downloads the UCI Daily and Sports Activities zip (~3 MB) and runs preprocessing (~5–10 min).
The processed arrays are cached to Google Drive so this step is **skipped on future sessions**.

In [ ]:
import os, sys, shutil, zipfile, urllib.request, json, numpy as np
from pathlib import Path

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

DSA_UCI_DIR   = f"{PROJECT_DIR}/UCI"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed/DSA"

# ── 1. Restore from Drive cache if available ─────────────────────────────────
cache_x = f"{DSA_CACHE_DIR}/X.npy"
if os.path.exists(cache_x):
    print("Restoring processed DSA from Drive cache...")
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    for fname in os.listdir(DSA_CACHE_DIR):
        shutil.copy(f"{DSA_CACHE_DIR}/{fname}", f"{PROCESSED_DIR}/{fname}")
    X = np.load(f"{PROCESSED_DIR}/X.npy")
    print(f"  Restored: X={X.shape}")
    print("DSA data ready ✅  (from Drive cache)")

# ── 2. Otherwise download from UCI + preprocess ───────────────────────────────
else:
    # Download
    UCI_URL  = "https://archive.ics.uci.edu/static/public/256/daily+and+sports+activities.zip"
    ZIP_PATH = "/tmp/dsa.zip"
    if not os.path.exists(ZIP_PATH):
        print("Downloading DSA dataset (~3 MB)...")
        urllib.request.urlretrieve(UCI_URL, ZIP_PATH)
        print(f"  Downloaded: {os.path.getsize(ZIP_PATH)/1e6:.1f} MB")
    else:
        print(f"Using cached download at {ZIP_PATH}")

    # Extract — zip contains data/a01/p1/s01.txt ...
    EXTRACT_DIR = "/tmp/dsa_extract"
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

    # Find the folder containing a01/, a02/ etc.
    data_root = None
    for root, dirs, files in os.walk(EXTRACT_DIR):
        if any(d.startswith("a0") for d in dirs):
            data_root = root
            break
    if data_root is None:
        raise RuntimeError(f"Could not find a0x folders inside {EXTRACT_DIR}")

    # Place at UCI/ (where download.py expects it)
    if os.path.exists(DSA_UCI_DIR):
        shutil.rmtree(DSA_UCI_DIR)
    shutil.copytree(data_root, DSA_UCI_DIR)
    print(f"  UCI/ ready: {len(list(Path(DSA_UCI_DIR).glob('a*')))} activity folders")

    # Preprocess
    print("\nPreprocessing DSA (~5–10 min)...")
    from src.data.preprocessing import preprocess_dataset
    data = preprocess_dataset("DSA", force=True)
    print(f"  X={data['X'].shape}, y={data['y'].shape}, subjects={data['subjects'].shape}")

    # Cache to Drive so future sessions skip this step
    print(f"\nSaving processed data to Drive cache ({DSA_CACHE_DIR})...")
    os.makedirs(DSA_CACHE_DIR, exist_ok=True)
    for fname in os.listdir(PROCESSED_DIR):
        shutil.copy(f"{PROCESSED_DIR}/{fname}", f"{DSA_CACHE_DIR}/{fname}")
    print("DSA data ready ✅  (cached to Drive for next session)")

## Step 5 — Smoke test

In [ ]:
import sys, os, subprocess, numpy as np
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

from src.data.preprocessing import preprocess_dataset
from src.models.factory import build_model, BASELINE_MODELS, TGNN_MODELS
from torch_geometric.data import Data
print("Imports ✅")

dsa = preprocess_dataset("DSA")
print(f"DSA   : X={dsa['X'].shape}, {len(np.unique(dsa['y']))} classes, {len(np.unique(dsa['subjects']))} subjects")

print(f"\nBaselines : {BASELINE_MODELS}")
print(f"TGNNs     : {TGNN_MODELS}")

# Dry-run to count pending jobs
r = subprocess.run(
    [sys.executable, "-m", "scripts.run_all", "--phase", "2", "--dry_run",
     "--results_dir", RESULTS_DIR],
    capture_output=True, text=True, cwd=PROJECT_DIR
)
lines = (r.stdout + r.stderr).splitlines()
jobs    = sum(1 for l in lines if "|" in l)
skipped = sum(1 for l in lines if "SKIP" in l)
print(f"\nJobs total : {jobs}  (already done: {skipped}, pending: {jobs - skipped})")

## Step 6 — Run Phase 2 🚀

Results are saved to Google Drive after every completed experiment.
If the session disconnects, re-run all cells from the top — done jobs will be skipped.

In [ ]:
import sys, os, subprocess
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

cmd = [
    sys.executable, "-u", "-m", "scripts.run_all",
    "--phase", "2",
    "--device", "cuda",
    "--results_dir", RESULTS_DIR,
]

print("Command:", " ".join(cmd))
print(f"Results → {RESULTS_DIR}/phase2/")
print(f"Log     → {LOG_PATH}")
print("=" * 60)

with open(LOG_PATH, "a") as logfile:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=PROJECT_DIR,
        env={**os.environ, "PYTHONPATH": PROJECT_DIR, "PYTHONUNBUFFERED": "1"},
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
        logfile.write(line)
        logfile.flush()

proc.wait()
print(f"\nProcess exited with code: {proc.returncode}")

## Step 7 — Check results

In [ ]:
import json
from pathlib import Path

p2_dir = Path(RESULTS_DIR) / "phase2"
complete, partial = [], []

for folder in sorted(p2_dir.glob("DSA_*")):
    if not folder.is_dir(): continue
    if (folder / "results.json").exists():
        complete.append(folder)
    elif any(folder.iterdir()):
        partial.append(folder.name)

print(f"Complete : {len(complete):3d} / 170")
print(f"Partial  : {len(partial):3d}  (interrupted — will be rerun next session)")
print()

for folder in complete:
    with open(folder / "results.json") as f:
        r = json.load(f)
    acc = r.get("accuracy_mean", r.get("test_accuracy", "?"))
    f1  = r.get("macro_f1_mean", r.get("macro_f1", "?"))
    line = f"  {folder.name:<45s}"
    if isinstance(acc, float):
        line += f"  acc={acc:.4f}  f1={f1:.4f}"
    print(line)

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `GITHUB_REPO` 404 / permission denied | Make repo public, or use a token URL: `https://TOKEN@github.com/user/repo.git` |
| `CUDA out of memory` | Reduce `batch_size` in `configs/phase2.yaml` on your Mac, re-push, re-clone |
| Session disconnects mid-run | Re-run all cells — completed jobs are skipped |
| `torch_scatter` / `torch_sparse` fails | Not required — ignore the warning |
| `Cannot locate UCI data` | Re-run Step 4 (the UCI download cell) |
| DSA preprocessing takes >15 min | Normal for first run; subsequent sessions use the Drive cache |

## Downloading results back to your Mac

```bash
# Using rclone (install: brew install rclone, then: rclone config)
rclone sync "gdrive:ThesisMaster/results/phase2" \
  "/Users/timele23/Library/Mobile Documents/com~apple~CloudDocs/Thesis_Master/results/phase2"

# Then run analysis locally:
cd "/Users/timele23/Library/Mobile Documents/com~apple~CloudDocs/Thesis_Master"
PYTHONPATH=$PWD python3 -m scripts.analyse_results
```